# M3 profile

Looks at the M2 database and prints what looks wrong. It changes nothing.
Run from top to bottom after `python src/load.py`.

In [1]:
import sys
sys.path.insert(0, "../src")   # so that `import db` works from notebooks/
import unicodedata
from datetime import date

import pandas as pd

import db

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

con = db.connect()
tables = {name: pd.read_sql_query(query, con) for name, query in db.TABLE_QUERIES.items()}


def title(text):
    print("\n" + "=" * 70 + "\n" + text + "\n" + "=" * 70)

## 1. rows

In [2]:
title("1. ROW COUNTS")
for name, df in tables.items():
    print("%-16s %8d rows" % (name, len(df)))


1. ROW COUNTS
genre                  23 rows
artist               1176 rows
album                1745 rows
track                2048 rows
recording             102 rows
track_detail          100 rows
album_detail          100 rows
chart_entry          2299 rows
artist_fans           100 rows
track_recording       102 rows
quarantine              0 rows


## 2. missing

In [3]:
title("2. MISSING VALUES (NULL or empty text)")
for name, df in tables.items():
    if name == "quarantine":
        continue
    for column in df.columns:
        missing = int(df[column].isna().sum() + (df[column] == "").sum())
        if missing:
            print("%s.%s: %d of %d (%.1f%%)" % (name, column, missing, len(df),
                                                100 * missing / len(df)))


2. MISSING VALUES (NULL or empty text)
recording.length_ms: 1 of 102 (1.0%)
recording.first_release_date: 9 of 102 (8.8%)


## 3. numbers

In [4]:
title("3. NUMBERS: min / median / max, zeros, negatives, most frequent value")
numeric = {"track": ["duration", "explicit"], "album_detail": ["nb_tracks"],
           "artist_fans": ["nb_fan"], "chart_entry": ["position", "rank"],
           "recording": ["length_ms"]}
for name, columns in numeric.items():
    for column in columns:
        s = tables[name][column].dropna()
        if s.empty:
            print("%s.%s: no values" % (name, column))
            continue
        top = s.value_counts().head(3).to_dict()
        print("%s.%s: min %s | median %s | max %s | zeros %d | negatives %d | most frequent %s"
              % (name, column, s.min(), s.median(), s.max(), (s == 0).sum(), (s < 0).sum(), top))


3. NUMBERS: min / median / max, zeros, negatives, most frequent value
track.duration: min 25 | median 207.5 | max 1638 | zeros 0 | negatives 0 | most frequent {192: 22, 212: 20, 224: 20}
track.explicit: min 0 | median 0.0 | max 1 | zeros 1814 | negatives 0 | most frequent {0: 1814, 1: 234}
album_detail.nb_tracks: min 1 | median 1.0 | max 45 | zeros 0 | negatives 0 | most frequent {1: 58, 12: 5, 16: 4}
artist_fans.nb_fan: min 597 | median 1148728.0 | max 20933670 | zeros 0 | negatives 0 | most frequent {1149727: 1, 11754475: 1, 5219785: 1}
chart_entry.position: min 1 | median 50.0 | max 100 | zeros 0 | negatives 0 | most frequent {1: 23, 2: 23, 3: 23}
chart_entry.rank: min 6963 | median 735846.0 | max 1000000 | zeros 0 | negatives 0 | most frequent {954234: 6, 954166: 5, 916433: 5}
recording.length_ms: min 120000.0 | median 206813.0 | max 364826.0 | zeros 0 | negatives 0 | most frequent {172533.0: 2, 184000.0: 2, 192000.0: 2}


## 4. dates

In [5]:
title("4. DATES")
today = date.today().isoformat()
for name, column in [("track_detail", "release_date"), ("album_detail", "release_date"),
                     ("recording", "first_release_date")]:
    s = tables[name][column].dropna()
    print("%s.%s: %s .. %s | in the future: %d | before 1950: %d"
          % (name, column, s.min() if len(s) else None, s.max() if len(s) else None,
             (s > today).sum(), (s < "1950").sum()))
days = tables["chart_entry"]["snapshot_date"]
print("chart snapshot days:", sorted(days.unique()))


4. DATES
track_detail.release_date: 1979-07-27 .. 2026-10-09 | in the future: 0 | before 1950: 0
album_detail.release_date: 1979-07-27 .. 2026-10-09 | in the future: 0 | before 1950: 0
recording.first_release_date: 1979-07-27 .. 2026-10-09 | in the future: 0 | before 1950: 0
chart snapshot days: ['2026-10-12']


## 5. text

In [6]:
title("5. TEXT: do equal things look equal?")
for name, column in [("artist", "name"), ("album", "title"), ("track", "title"), ("genre", "name")]:
    s = tables[name][column].dropna()
    not_nfc = s[s != s.map(lambda x: unicodedata.normalize("NFC", x))]
    padded = s[s != s.str.strip()]
    nbsp = s[s.str.contains("\u00a0", regex=False)]
    double = s[s.str.contains("  ", regex=False)]
    print("%s.%s: not NFC %d | leading/trailing space %d | non-breaking space %d | double space %d"
          % (name, column, len(not_nfc), len(padded), len(nbsp), len(double)))
    for label, sample in [("not NFC", not_nfc), ("padded", padded), ("nbsp", nbsp)]:
        for value in sample.head(3):
            print("    %s: %r" % (label, value))


5. TEXT: do equal things look equal?
artist.name: not NFC 0 | leading/trailing space 3 | non-breaking space 1 | double space 0
    padded: 'ILLIT '
    padded: 'Leandro & Leonardo '
    padded: 'Dhanush '
    nbsp: 'Çocuk şarkıları\xa0Superstar'
album.title: not NFC 1 | leading/trailing space 1 | non-breaking space 1 | double space 0
    not NFC: 'Fille à Papa'
    padded: 'Bak postacı geliyor\xa0'
    nbsp: 'Bak postacı geliyor\xa0'
track.title: not NFC 3 | leading/trailing space 2 | non-breaking space 2 | double space 0
    not NFC: 'Fille à Papa'
    not NFC: 'Çok Güzel Gülüyorsun'
    not NFC: 'Adaletin Bu Mu Dünya'
    padded: 'Bak postacı geliyor\xa0'
    padded: "Al 1'de Burdan Yak\xa0"
    nbsp: 'Bak postacı geliyor\xa0'
    nbsp: "Al 1'de Burdan Yak\xa0"
genre.name: not NFC 0 | leading/trailing space 0 | non-breaking space 0 | double space 0


## 6. identity

In [7]:
title("6. IDENTITY: one thing recorded as two?")
artist = tables["artist"].copy()
artist["same_name"] = artist["name"].str.casefold().str.strip()
dup = artist[artist.duplicated("same_name", keep=False) & artist["same_name"].notna()]
print("artists with the same name but different Deezer id:", dup["same_name"].nunique(), "names")
print(dup.sort_values("same_name")[["deezer_id", "name"]].head(15).to_string(index=False))
track = tables["track"].copy()
track["same_title"] = track["title"].str.casefold().str.strip()
dup = track[track.duplicated(["same_title", "artist_key"], keep=False)]
print("\ntracks with the same title and artist but different track id:",
      len(dup), "rows")
print(dup.sort_values("same_title")[["deezer_id", "title", "artist_key"]].head(10).to_string(index=False))
isrc = tables["track_detail"].dropna(subset=["isrc"])
shared = isrc[isrc.duplicated("isrc", keep=False)]
print("\nISRCs shared by more than one track:", shared["isrc"].nunique())


6. IDENTITY: one thing recorded as two?
artists with the same name but different Deezer id: 2 names
 deezer_id  name
  11446160  LiSA
 145068682  LISA
   8673550 Vacra
 222719305 Vacra

tracks with the same title and artist but different track id: 13 rows
 deezer_id                                           title  artist_key
4015341011                                         Dai Dai           1
4095595211                                         Dai Dai           1
4078687321                                         Dai Dai           1
    576431                                Eye of the Tiger         637
    632868                                Eye of the Tiger         637
 140869461                                           Human          65
 128878464                                           Human          65
  14552280 My Heart Will Go On (Love Theme from "Titanic")         109
  17000055 My Heart Will Go On (Love Theme from "Titanic")         109
  64920072                       

## 7. coverage

In [8]:
title("7. CHART COVERAGE: equal things compared equally?")
chart = tables["chart_entry"]
per_chart = chart.groupby(["genre_key", "snapshot_date"]).size().rename("rows").reset_index()
print("rows per (genre, day): min %d | median %d | max %d"
      % (per_chart["rows"].min(), per_chart["rows"].median(), per_chart["rows"].max()))
print("charts with fewer than 100 rows:", int((per_chart["rows"] < 100).sum()), "of", len(per_chart))
print("days per genre:", chart.groupby("genre_key")["snapshot_date"].nunique().describe()[["min", "max"]].to_dict())
same_chart = chart.groupby(["genre_key", "snapshot_date", "track_key"]).size()
print("tracks in more than one position of the same chart:", int((same_chart > 1).sum()))
have = set(tables["artist_fans"]["artist_key"])
print("artists that appear in charts but have no fan count:",
      len(set(tables["artist"]["artist_key"]) - have), "of", len(tables["artist"]))


7. CHART COVERAGE: equal things compared equally?
rows per (genre, day): min 99 | median 100 | max 100
charts with fewer than 100 rows: 1 of 23
days per genre: {'min': 1.0, 'max': 1.0}
tracks in more than one position of the same chart: 0
artists that appear in charts but have no fan count: 1076 of 1176


## 8. sequences

In [9]:
title("8. SEQUENCES: only visible over time")
chart = chart.sort_values(["genre_key", "track_key", "snapshot_date"])
chart["jump"] = chart.groupby(["genre_key", "track_key"])["position"].diff().abs()
print("largest position jumps between two days:")
print(chart.nlargest(5, "jump")[["genre_key", "track_key", "snapshot_date", "position", "jump"]]
      .to_string(index=False))
stuck = chart.groupby(["genre_key", "track_key"])["rank"].agg(["nunique", "count"])
print("tracks whose rank is identical on 3 or more days:",
      int(((stuck["nunique"] == 1) & (stuck["count"] >= 3)).sum()))
fans = tables["artist_fans"].sort_values(["artist_key", "snapshot_date"])
fans["change"] = fans.groupby("artist_key")["nb_fan"].diff()
print("artists whose fan count decreased between two days:", int((fans["change"] < 0).sum()),
      "| biggest decrease:", fans["change"].min())


8. SEQUENCES: only visible over time
largest position jumps between two days:
 genre_key  track_key snapshot_date  position  jump
         1          1    2026-10-12         1   NaN
         1          2    2026-10-12         2   NaN
         1          3    2026-10-12         3   NaN
         1          4    2026-10-12         4   NaN
         1          5    2026-10-12         5   NaN
tracks whose rank is identical on 3 or more days: 0
artists whose fan count decreased between two days: 0 | biggest decrease: nan


## 9. quarantine

In [10]:
title("9. QUARANTINE (records M2 could not load)")
print(tables["quarantine"].groupby(["source", "reason"]).size().rename("how_many").to_string())


9. QUARANTINE (records M2 could not load)
Series([], )
